# Prajna CRN Inference

Load trained CRN checkpoint on Gemma 4 E2B and run interactive generation.

**Requirements:** Runtime → Change runtime type → T4 GPU

**Setup:** Upload `prajna_checkpoints.zip` when prompted in Cell 2.

In [ ]:
# Cell 1: Install Dependencies
!pip install -q torch transformers accelerate einops huggingface_hub

import torch, os, json, time, glob, zipfile
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F

BASE = '/content/prajna'
CKPT_DIR = f'{BASE}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2: Upload checkpoint zip (skips if already extracted)
ckpt_files = sorted(glob.glob(f'{CKPT_DIR}/*.pt'))
if ckpt_files:
    print(f'Checkpoints already extracted ({len(ckpt_files)} files). Skipping upload.')
else:
    from google.colab import files
    print('Upload prajna_checkpoints.zip')
    uploaded = files.upload()
    zip_path = list(uploaded.keys())[0]
    print(f'Uploaded: {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content')
    print(f'Checkpoints extracted to {CKPT_DIR}')
print(sorted(glob.glob(f'{CKPT_DIR}/*.pt')))

In [ ]:
# Cell 3: CRN Components (must match training notebook exactly)

class ResonanceAttention(nn.Module):
    def __init__(self, d_model, num_heads=4, num_frequencies=16, top_k=4):
        super().__init__()
        self.num_heads = num_heads
        self.num_frequencies = num_frequencies
        self.top_k = top_k
        self.head_dim = d_model // num_heads
        self.freq_q = nn.Linear(d_model, num_heads * num_frequencies, bias=False)
        self.freq_k = nn.Linear(d_model, num_heads * num_frequencies, bias=False)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        device = x.device
        q = self.freq_q(x).view(B, T, self.num_heads, self.num_frequencies)
        k = self.freq_k(x).view(B, T, self.num_heads, self.num_frequencies)
        freq_scores = F.softmax(q, dim=-1)
        top_freq_vals, top_freq_idx = freq_scores.topk(min(self.top_k, self.num_frequencies), dim=-1)
        top_freq_vals = top_freq_vals / (top_freq_vals.sum(dim=-1, keepdim=True) + 1e-8)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim)
        out = torch.zeros_like(v)
        for f_idx in range(self.num_frequencies):
            mask = (top_freq_idx == f_idx).any(dim=-1)
            if mask.sum() == 0:
                continue
            freq_weight = torch.zeros(B, T, self.num_heads, device=device)
            for k_idx in range(self.top_k):
                match = (top_freq_idx[:, :, :, k_idx] == f_idx)
                freq_weight += match.float() * top_freq_vals[:, :, :, k_idx]
            q_f = q[:, :, :, f_idx]
            k_f = k[:, :, :, f_idx]
            attn_scores = torch.einsum('bih,bjh->bhij', q_f, k_f) / (self.head_dim ** 0.5)
            attn_mask = mask.unsqueeze(2) * mask.unsqueeze(1)
            attn_scores = attn_scores.masked_fill(~attn_mask.permute(0, 3, 1, 2).bool(), float('-inf'))
            attn_weights = F.softmax(attn_scores, dim=-1).nan_to_num(0.0)
            out += freq_weight.unsqueeze(-1) * torch.einsum('bhij,bjhd->bihd', attn_weights, v)
        return self.out_proj(out.reshape(B, T, D))

class EpisodicMemory:
    def __init__(self, d_model, mem_size=512, mem_dim=128, device='cuda'):
        self.mem_size = mem_size
        self.mem_dim = mem_dim
        self.d_model = d_model
        self.device = device
        self.memory = torch.zeros(mem_size, mem_dim, device=device)
        self.temporal_positions = torch.zeros(mem_size, device=device)
        self.write_ptr = 0
        self.step_count = 0
        self.compress = nn.Linear(d_model, mem_dim).to(device)
        self.decompress = nn.Linear(mem_dim, d_model).to(device)
        self.read_gate = nn.Linear(d_model, mem_dim).to(device)
        self.write_gate = nn.Linear(d_model, 1).to(device)
        self.relevance_gate = nn.Linear(d_model + mem_dim, 1).to(device)

    def get_parameters(self):
        return (list(self.compress.parameters()) + list(self.decompress.parameters()) +
                list(self.read_gate.parameters()) + list(self.write_gate.parameters()) +
                list(self.relevance_gate.parameters()))

    def read(self, query, top_k=8):
        if query.dim() == 1:
            query = query.unsqueeze(0)
        B = query.shape[0]
        q_compressed = self.read_gate(query)
        mem_expanded = self.memory.unsqueeze(0).expand(B, -1, -1)
        q_norm = F.normalize(q_compressed, dim=-1)
        mem_norm = F.normalize(mem_expanded, dim=-1)
        sims = torch.bmm(q_norm.unsqueeze(1), mem_norm.transpose(1, 2)).squeeze(1)
        recency = self.temporal_positions / (self.temporal_positions.max() + 1)
        sims = sims + 0.1 * recency.unsqueeze(0)
        top_k = min(top_k, self.mem_size)
        top_vals, top_idx = sims.topk(top_k, dim=-1)
        attn_weights = F.softmax(top_vals, dim=-1)
        retrieved = torch.gather(mem_expanded, 1, top_idx.unsqueeze(-1).expand(-1, -1, self.mem_dim))
        retrieved = (retrieved * attn_weights.unsqueeze(-1)).sum(dim=1)
        return self.decompress(retrieved), attn_weights

    def write(self, content, force=False):
        gate_value = torch.sigmoid(self.write_gate(content.unsqueeze(0))).item()
        if gate_value < 0.5 and not force:
            return False
        compressed = self.compress(content.detach())
        if self.write_ptr < self.mem_size:
            slot = self.write_ptr
            self.write_ptr += 1
        else:
            slot = self.temporal_positions.argmin().item()
        write_weight = min(gate_value, 0.9)
        self.memory[slot] = (write_weight * compressed + (1 - write_weight) * self.memory[slot].clone()).detach()
        self.step_count += 1
        self.temporal_positions[slot] = self.step_count
        return True

    def save(self, path):
        state = {
            'memory': self.memory.detach().cpu().float().numpy().tolist(),
            'temporal_positions': self.temporal_positions.detach().cpu().float().numpy().tolist(),
            'write_ptr': self.write_ptr,
            'step_count': self.step_count
        }
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        with open(path, 'w') as f:
            json.dump(state, f)

    def load(self, path):
        with open(path) as f:
            state = json.load(f)
        self.memory = torch.tensor(state['memory'], dtype=torch.float32, device=self.device)
        self.temporal_positions = torch.tensor(state['temporal_positions'], dtype=torch.float32, device=self.device)
        self.write_ptr = state['write_ptr']
        self.step_count = state['step_count']

    def get_stats(self):
        return {
            'used_slots': (self.temporal_positions > 0).sum().item(),
            'total_slots': self.mem_size,
            'write_ptr': self.write_ptr,
            'step_count': self.step_count,
        }

class ReflectiveLoop(nn.Module):
    def __init__(self, d_model, num_corrections=16):
        super().__init__()
        self.num_corrections = num_corrections
        self.d_model = d_model
        self.critic = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.GELU(),
            nn.Linear(d_model // 4, num_corrections + 1)
        )
        self.correction_directions = nn.Parameter(torch.randn(num_corrections, d_model) * 0.01)
        self.thresholds = nn.Parameter(torch.ones(num_corrections) * 0.5)
        self.confidence_scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, hidden_state, return_correction_id=False):
        pooled = hidden_state.mean(dim=1) if hidden_state.dim() == 3 else hidden_state
        scores = self.critic(pooled)
        no_correction_score = scores[:, -1]
        correction_scores = scores[:, :-1]
        best_score, best_idx = correction_scores.max(dim=-1)
        apply_correction = best_score > (no_correction_score + 0.2)
        corrected_state = hidden_state.clone()
        correction_id = -1
        if apply_correction.any():
            for b in range(hidden_state.shape[0]):
                if apply_correction[b]:
                    correction = self.correction_directions[best_idx[b]]
                    confidence = torch.sigmoid(best_score[b] - self.thresholds[best_idx[b]])
                    scale = torch.abs(self.confidence_scale)
                    corrected_state[b] = hidden_state[b] + scale * confidence * correction
                    correction_id = best_idx[b].item()
        return (corrected_state, correction_id) if return_correction_id else corrected_state

class SkillComposer(nn.Module):
    def __init__(self, d_model, num_skills=64, skill_rank=8, top_k=4):
        super().__init__()
        self.num_skills = num_skills
        self.skill_rank = skill_rank
        self.top_k = top_k
        self.d_model = d_model
        self.skill_u = nn.Parameter(torch.randn(num_skills, d_model, skill_rank) * 0.01)
        self.skill_v = nn.Parameter(torch.randn(num_skills, skill_rank, d_model) * 0.01)
        self.router = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.GELU(),
            nn.Linear(d_model // 4, num_skills)
        )
        self.skill_scale = nn.Parameter(torch.ones(num_skills) * 0.01)

    def forward(self, x):
        B, T, D = x.shape
        skill_logits = self.router(x.mean(dim=1))
        skill_weights = F.softmax(skill_logits, dim=-1)
        top_k = min(self.top_k, self.num_skills)
        top_weights, top_indices = skill_weights.topk(top_k, dim=-1)
        top_weights = top_weights / (top_weights.sum(dim=-1, keepdim=True) + 1e-8)
        perturbation = torch.zeros_like(x)
        for k in range(self.top_k):
            u = self.skill_u[top_indices[:, k]]
            v = self.skill_v[top_indices[:, k]]
            scale = torch.abs(self.skill_scale[top_indices[:, k]])
            x_v = torch.bmm(x, v.transpose(1, 2))
            perturbation += top_weights[:, k].unsqueeze(1).unsqueeze(-1) * scale.unsqueeze(1).unsqueeze(-1) * torch.bmm(x_v, u.transpose(1, 2))
        return x + perturbation

print('CRN components loaded')

In [ ]:
# Cell 4: PrajnaStudent — CRN as post-hoc adapter on final hidden state
from transformers import AutoModelForCausalLM, AutoTokenizer

class PrajnaStudent(nn.Module):
    def __init__(self, device='cuda'):
        super().__init__()
        self.device = device
        import gc
        torch.cuda.empty_cache()
        gc.collect()
        print(f'VRAM before load: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
        print('Loading E2B student...')
        self.tok = AutoTokenizer.from_pretrained('google/gemma-4-E2B')
        self.base_model = AutoModelForCausalLM.from_pretrained(
            'google/gemma-4-E2B', torch_dtype=torch.float16, device_map='auto'
        )
        for p in self.base_model.parameters():
            p.requires_grad = False
        self.vocab = 262144
        self.d_model = 1536
        self.mem = EpisodicMemory(self.d_model, mem_size=512, mem_dim=128, device=device)
        self.reflection = ReflectiveLoop(d_model=self.d_model, num_corrections=16).to(device)
        self.skills = SkillComposer(d_model=self.d_model, num_skills=64, skill_rank=8, top_k=4).to(device)
        self.resonance = ResonanceAttention(d_model=self.d_model, num_heads=4, num_frequencies=16, top_k=4).to(device)
        crn_params = self.get_params()
        total = sum(p.numel() for p in crn_params)
        print(f'CRN: {total:,} params (float32)')

    def forward(self, input_ids):
        with torch.no_grad():
            hidden = self.base_model.model.language_model(input_ids).last_hidden_state
        h = hidden.to(torch.float32)
        r = self.resonance(h)
        s = self.skills(h)
        h = h + r + s
        if self.mem.temporal_positions.sum() > 0:
            read_out, _ = self.mem.read(h.mean(dim=1).to(torch.float32), top_k=8)
            h = h + read_out.unsqueeze(1)
        logits = self.base_model.lm_head(h.to(torch.float16))
        return logits

    def get_params(self):
        return (self.mem.get_parameters() + list(self.reflection.parameters()) +
                list(self.skills.parameters()) + list(self.resonance.parameters()))

    def load_memory(self, p):
        self.mem.load(p)

print('Student class defined')

In [ ]:
# Cell 5: Load trained model and checkpoint
device = 'cuda'
student = PrajnaStudent(device=device)
student.eval()

ckpt_file = f'{CKPT_DIR}/dpo_final.pt'
if not os.path.exists(ckpt_file):
    ckpt_file = sorted(glob.glob(f'{CKPT_DIR}/*_final.pt'))[-1]
print(f'Loading: {ckpt_file}')
ckpt = torch.load(ckpt_file, map_location=device, weights_only=False)
crn_state = {k: v for k, v in ckpt['crn'].items()}
student.resonance.load_state_dict({k.replace('resonance.', ''): v for k,v in crn_state.items() if k.startswith('resonance.')})
student.skills.load_state_dict({k.replace('skills.', ''): v for k,v in crn_state.items() if k.startswith('skills.')})
student.reflection.load_state_dict({k.replace('reflection.', ''): v for k,v in crn_state.items() if k.startswith('reflection.')})
mem_state = {k.replace('mem.', ''): v for k,v in crn_state.items() if k.startswith('mem.')}
if mem_state:
    student.mem.compress.load_state_dict({'weight': mem_state['compress.weight'], 'bias': mem_state['compress.bias']})
    student.mem.decompress.load_state_dict({'weight': mem_state['decompress.weight'], 'bias': mem_state['decompress.bias']})
    student.mem.read_gate.load_state_dict({'weight': mem_state['read_gate.weight'], 'bias': mem_state['read_gate.bias']})
    student.mem.write_gate.load_state_dict({'weight': mem_state['write_gate.weight'], 'bias': mem_state['write_gate.bias']})
    student.mem.relevance_gate.load_state_dict({'weight': mem_state['relevance_gate.weight'], 'bias': mem_state['relevance_gate.bias']})

mem_file = ckpt.get('memory_file', f'{CKPT_DIR}/memory_dpo_final.json')
if os.path.exists(mem_file):
    student.load_memory(mem_file)
    print(f'Memory loaded: {student.mem.get_stats()}')

print(f'Step: {ckpt["step"]} | Loss: {ckpt["loss"]:.4f}')
print(f'VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
print('Model ready!')

In [ ]:
# Cell 6: Interactive generation
@torch.no_grad()
def generate(prompt, max_new_tokens=64, temperature=0.6, repetition_penalty=1.2, top_k=50, top_p=0.9):
    enc = student.tok(prompt, return_tensors='pt').to(device)
    input_ids = enc['input_ids']

    for _ in range(max_new_tokens):
        logits = student(input_ids)
        next_logits = logits[:, -1, :] / temperature
        # Repetition penalty
        if repetition_penalty != 1.0:
            for token_id in set(input_ids[0].tolist()):
                next_logits[:, token_id] /= repetition_penalty

        # Top-k filtering
        if top_k > 0:
            top_k_vals, _ = torch.topk(next_logits, top_k, dim=-1)
            threshold = top_k_vals[:, -1].unsqueeze(-1)
            next_logits[next_logits < threshold] = float('-inf')

        # Top-p (nucleus) filtering
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(next_logits, descending=True, dim=-1)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
            sorted_indices_to_remove[:, 0] = False
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
            next_logits[indices_to_remove] = float('-inf')

        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=1)
        if next_token.item() == student.tok.eos_token_id:
            break

    return student.tok.decode(input_ids[0], skip_special_tokens=True)

test_prompts = [
    'Explain resonance in physics.',
    'Write a short poem about AI.',
    'What is deep learning?',
    'Describe the water cycle.',
    'Why is the sky blue?',
]

print('=' * 60)
print('CRN GENERATION TEST')
print('=' * 60)
for prompt in test_prompts:
    print(f'\nPrompt: {prompt}')
    print('-' * 60)
    out = generate(prompt, max_new_tokens=64)
    print(f'{out}')
    print()

# Interactive prompt
print('\n' + '=' * 60)
print('Try your own prompt:')
user_prompt = input('Enter a prompt: ')  # noqa: F821
if user_prompt.strip():
    out = generate(user_prompt.strip(), max_new_tokens=128)
    print(f'\n{out}')

In [ ]:
# Cell 7: Memory analysis
stats = student.mem.get_stats()
print('Episodic Memory State:')
print(f'  Slots used: {stats["used_slots"]}/{stats["total_slots"]}')
print(f'  Write pointer: {stats["write_ptr"]}')
print(f'  Total writes: {stats["step_count"]}')

# Memory content summary
norm_per_slot = torch.norm(student.mem.memory, dim=-1)
print(f'  Memory norm: mean={norm_per_slot.mean():.4f}, std={norm_per_slot.std():.4f}')

# Temporal distribution
tp = student.mem.temporal_positions
active = tp > 0
print(f'  Temporal positions: min={tp[active].min().item():.0f}, max={tp[active].max().item():.0f}')

# Reflection stats
if hasattr(student.reflection, 'get_correction_stats'):
    print(f'\nReflective Loop:')
    print(f'  {student.reflection.get_correction_stats()}')

In [ ]:
# Cell 8: Baseline comparison (load base model without CRN)
print('Loading baseline (no CRN)...')
baseline = AutoModelForCausalLM.from_pretrained(
    'google/gemma-4-E2B', torch_dtype=torch.float16, device_map='auto'
)
baseline.eval()

print(f'\nBaseline VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

# Compare generations
for prompt in ['Explain AI.', 'What is gravity?']:
    enc = student.tok(prompt, return_tensors='pt').to(device)
    input_ids = enc['input_ids']

    with torch.no_grad():
        out_base = baseline.generate(input_ids, max_new_tokens=48, temperature=0.7)
    base_text = student.tok.decode(out_base[0], skip_special_tokens=True)

    crn_text = generate(prompt, max_new_tokens=48)

    print(f'\nPrompt: {prompt}')
    print('  Baseline:', base_text[:200])
    print('  +CRN:    ', crn_text[:200])

del baseline
torch.cuda.empty_cache()
import gc; gc.collect()

In [ ]:
# Cell 9: Memory persistence test — does CRN recall across sessions?
print('Memory Persistence Test')
print('=' * 60)

# Write a fact to memory
fact = 'My favorite color is teal and I live in Tokyo.'
enc = student.tok(fact, return_tensors='pt').to(device)
with torch.no_grad():
    _ = student(enc['input_ids'])
    h = student.base_model.model.language_model(enc['input_ids']).last_hidden_state
    student.mem.write(h[:, -1, :].mean(dim=0).to(torch.float32), force=True)
print(f'Written: "{fact}"')
print(f'Memory now: {student.mem.get_stats()}')

# Query
query = 'What is my favorite color and where do I live?'
out = generate(query, max_new_tokens=32)
print(f'\nQuery: {query}')
print(f'CRN:   {out}')